## PHASE 1: SYNTHETIC DATA GENERATION

In [1]:
# Import and setup

!pip install faker

In [2]:
import random
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

In [3]:
# Define the fields the dataset needs

N_SUBJECTS = 300
SITES = ["SITE_01", "SITE_02", "SITE_03", "SITE_04", "SITE_05"]
VISIT_LABELS = ["Screening", "Baseline", "Week4", "Week8", "Week12"]

LAB_RANGES = {
    "hemoglobin_g_dl": (12.0, 17.0),
    "wbc_count_10e9_l": (4.0, 11.0),
    "alt_u_l": (7, 56),
    "systolic_bp_mmhg": (90, 140),
}

In [4]:
# Create a helper function for the random dates, and the subjects ID list

def random_date(start, end):
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

study_start = datetime(2025, 1, 1)
study_end = datetime(2025, 12, 31)

subject_ids = [f"SUBJ-{1000+i}" for i in range(N_SUBJECTS)]

In [5]:
# Generate each subject's visit rows

rows = []

for subj in subject_ids:
    site = random.choice(SITES)
    n_visits = random.choice([3, 4, 5])
    visit_dates = sorted([random_date(study_start, study_end) for _ in range(n_visits)])

    for i, visit_date in enumerate(visit_dates):
        visit_label = VISIT_LABELS[i] if i < len(VISIT_LABELS) else f"Visit{i+1}"

        row = {
            "subject_id": subj,
            "site_id": site,
            "visit_label": visit_label,
            "visit_date": visit_date.strftime("%Y-%m-%d"),
            "age": random.randint(18, 75),
            "sex": random.choice(["M", "F"]),
        }

        for lab, (low, high) in LAB_RANGES.items():
            row[lab] = round(np.random.uniform(low, high), 1)

        row["adverse_event_flag"] = np.random.choice([0, 1], p=[0.9, 0.1])

        rows.append(row)

len(rows)

1206

In [6]:
# Convert the dataset into a table and insert data-quality problems on purpose

df = pd.DataFrame(rows)

# 1. Missing values (~4% of lab cells)
for lab in LAB_RANGES:
    mask = np.random.rand(len(df)) < 0.04
    df.loc[mask, lab] = np.nan

# 2. Out-of-range lab values (~3% of rows)
n_outliers = int(len(df) * 0.03)
outlier_idx = np.random.choice(df.index, n_outliers, replace=False)
for idx in outlier_idx:
    lab = random.choice(list(LAB_RANGES.keys()))
    low, high = LAB_RANGES[lab]
    df.loc[idx, lab] = round(high * random.uniform(2.5, 4.0), 1)

# 3. Duplicate rows (~1.5%)
n_duplicates = int(len(df) * 0.015)
duplicate_rows = df.sample(n_duplicates, random_state=1).copy()
df = pd.concat([df, duplicate_rows], ignore_index=True)

# 4. Add a few impossible ages
bad_age_idx = np.random.choice(df.index, 3, replace=False)
df.loc[bad_age_idx, "age"] = random.choice([0, 150, -5])

df.shape

(1224, 11)

In [7]:
# View the table, shuffle it, and save it as a csv file

df = df.sample(frac=1, random_state=7).reset_index(drop=True)

df.head()

,subject_id,site_id,visit_label,visit_date,age,sex,hemoglobin_g_dl,wbc_count_10e9_l,alt_u_l,systolic_bp_mmhg,adverse_event_flag
0,SUBJ-1013,SITE_02,Week4,2025-10-18,66,M,14.8,10.6,41.1,118.5,0
1,SUBJ-1131,SITE_03,Screening,2025-03-16,26,M,16.2,4.0,19.1,127.0,0
2,SUBJ-1045,SITE_04,Screening,2025-06-02,72,F,13.0,6.1,50.9,90.7,0
3,SUBJ-1291,SITE_02,Baseline,2025-03-06,30,M,14.3,7.6,7.4,111.1,0
4,SUBJ-1231,SITE_04,Week4,2025-12-01,40,M,14.0,5.0,7.5,117.2,1


In [8]:
df.to_csv("clinical_trial_synthetic.csv", index=False)

### PHASE 2: SQL DATA-QUALITY ANALYSIS

In [9]:
# Load the CSV into a SQLite database

import sqlite3

conn = sqlite3.connect("clinical_trial.db")

df_loaded = pd.read_csv("clinical_trial_synthetic.csv")

df_loaded.to_sql("trial_data", conn, if_exists="replace", index=False)

pd.read_sql("SELECT * FROM trial_data LIMIT 5;", conn)

,subject_id,site_id,visit_label,visit_date,age,sex,hemoglobin_g_dl,wbc_count_10e9_l,alt_u_l,systolic_bp_mmhg,adverse_event_flag
0,SUBJ-1013,SITE_02,Week4,2025-10-18,66,M,14.8,10.6,41.1,118.5,0
1,SUBJ-1131,SITE_03,Screening,2025-03-16,26,M,16.2,4.0,19.1,127.0,0
2,SUBJ-1045,SITE_04,Screening,2025-06-02,72,F,13.0,6.1,50.9,90.7,0
3,SUBJ-1291,SITE_02,Baseline,2025-03-06,30,M,14.3,7.6,7.4,111.1,0
4,SUBJ-1231,SITE_04,Week4,2025-12-01,40,M,14.0,5.0,7.5,117.2,1


In [10]:
# Query 1 : Check for the missing values by sites

query1 = """
SELECT 
    site_id, 
    COUNT(*) AS total_records,
    SUM(CASE WHEN hemoglobin_g_dl IS NULL THEN 1 ELSE 0 END) AS missing_hemoglobin,
    SUM(CASE WHEN wbc_count_10e9_l IS NULL THEN 1 ELSE 0 END) AS missing_wbc,
    SUM(CASE WHEN alt_u_l IS NULL THEN 1 ELSE 0 END) AS missing_alt,
    SUM(CASE WHEN systolic_bp_mmhg IS NULL THEN 1 ELSE 0 END) AS missing_bp
FROM trial_data
GROUP BY site_id
ORDER BY site_id
"""
pd.read_sql(query1, conn)

,site_id,total_records,missing_hemoglobin,missing_wbc,missing_alt,missing_bp
0,SITE_01,227,8,14,9,10
1,SITE_02,242,6,8,11,8
2,SITE_03,274,15,13,6,12
3,SITE_04,211,6,5,10,13
4,SITE_05,270,9,15,10,11


In [11]:
# Query2: Check for the Out-of-range lab values

query2 = """
SELECT
    subject_id,
    site_id, 
    visit_label,
    hemoglobin_g_dl,
    wbc_count_10e9_l,
    alt_u_l,
    systolic_bp_mmhg
FROM trial_data
WHERE hemoglobin_g_dl NOT BETWEEN 12.0 AND 17.0
    OR wbc_count_10e9_l NOT BETWEEN 4.0 AND 11.0
    OR alt_u_l NOT BETWEEN 7 AND 56
    OR systolic_bp_mmhg NOT BETWEEN 90 AND 140
ORDER BY subject_id ASC;
"""

pd.read_sql(query2, conn)

,subject_id,site_id,visit_label,hemoglobin_g_dl,wbc_count_10e9_l,alt_u_l,systolic_bp_mmhg
0,SUBJ-1008,SITE_04,Week8,13.8,8.4,216.2,116.8
1,SUBJ-1012,SITE_05,Week8,15.6,38.2,8.2,122.3
2,SUBJ-1022,SITE_01,Week8,15.5,4.5,47.3,413.5
3,SUBJ-1023,SITE_04,Week12,13.9,6.0,187.2,NaN
4,SUBJ-1025,SITE_02,Screening,16.5,4.3,20.8,387.3
5,SUBJ-1025,SITE_02,Week4,54.8,8.1,10.8,138.7
6,SUBJ-1030,SITE_05,Baseline,14.6,9.5,155.8,121.1
7,SUBJ-1041,SITE_05,Week8,16.0,4.0,165.1,109.9
8,SUBJ-1045,SITE_04,Baseline,13.0,4.2,199.9,119.2
9,SUBJ-1051,SITE_03,Week4,55.5,10.0,48.4,106.0


In [12]:
# Query3: Check for duplicate subject/visit records

query3 = """
SELECT 
    subject_id,
    visit_label,
    COUNT(*) AS number_of_records
FROM trial_data
GROUP BY subject_id, visit_label
HAVING COUNT(*) > 1;
"""

pd.read_sql(query3, conn)

,subject_id,visit_label,number_of_records
0,SUBJ-1013,Screening,2
1,SUBJ-1047,Week12,2
2,SUBJ-1047,Week8,2
3,SUBJ-1049,Week8,2
4,SUBJ-1050,Screening,2
5,SUBJ-1066,Baseline,2
6,SUBJ-1077,Week4,2
7,SUBJ-1078,Screening,2
8,SUBJ-1082,Week4,2
9,SUBJ-1082,Week8,2


In [13]:
# Query4: Adverse event rate by site

query4 = """
SELECT
    site_id,
    COUNT(*) AS total_visits,
    SUM(adverse_event_flag) AS total_adverse_events,
    ROUND(100.0 * SUM(adverse_event_flag) / COUNT(*), 1) AS adverse_event_rate_pct
FROM trial_data
GROUP BY site_id
ORDER BY adverse_event_rate_pct DESC;
"""
pd.read_sql(query4, conn)

,site_id,total_visits,total_adverse_events,adverse_event_rate_pct
0,SITE_01,227,25,11.0
1,SITE_02,242,26,10.7
2,SITE_03,274,27,9.9
3,SITE_04,211,18,8.5
4,SITE_05,270,20,7.4


In [14]:
# Query5: Visit completeness by subject

query5 = """
SELECT 
    subject_id,
    COUNT(DISTINCT visit_label) AS visits_completed
FROM trial_data
GROUP BY subject_id
ORDER BY visits_completed ASC
LIMIT 15;
"""

pd.read_sql(query5, conn)

,subject_id,visits_completed
0,SUBJ-1000,3
1,SUBJ-1001,3
2,SUBJ-1003,3
3,SUBJ-1005,3
4,SUBJ-1007,3
5,SUBJ-1009,3
6,SUBJ-1011,3
7,SUBJ-1013,3
8,SUBJ-1015,3
9,SUBJ-1016,3


In [15]:
# Query6: Visit date sequence errors

query6 = """
SELECT
    a.subject_id, a.visit_label AS earlier_visit, a.visit_date AS earlier_date,
    b.visit_label AS later_visit, b.visit_date AS later_date
FROM trial_data a
JOIN trial_data b
    ON a.subject_id = b.subject_id
    AND a.visit_date > b.visit_date
    AND a.visit_label IN ('Screening')
    AND b.visit_label IN ('Baseline', 'Week4', 'Week8', 'Week12');
"""

pd.read_sql(query6, conn)

,subject_id,earlier_visit,earlier_date,later_visit,later_date


In [16]:
# Query7: Sex distribution by site

query7 = """
SELECT 
    site_id,
    SUM(CASE WHEN sex = 'M' THEN 1 ELSE 0 END) AS male_count,
    SUM(CASE WHEN sex = 'F' THEN 1 ELSE 0 END) AS female_count,
    COUNT(*) AS total_subjects,
    ROUND(100.0 * SUM(CASE WHEN sex = 'M' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_male,
    ROUND(100.0 * SUM(CASE WHEN sex = 'F' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_female
FROM (
    SELECT DISTINCT subject_id, site_id, sex
    FROM trial_data
    ) AS unique_subjects
GROUP BY site_id
ORDER BY site_id;
"""

pd.read_sql(query7, conn)


,site_id,male_count,female_count,total_subjects,pct_male,pct_female
0,SITE_01,54,55,109,49.5,50.5
1,SITE_02,53,57,110,48.2,51.8
2,SITE_03,62,62,124,50.0,50.0
3,SITE_04,48,48,96,50.0,50.0
4,SITE_05,62,62,124,50.0,50.0


## PHASE 3: AI-ASSISTED ANOMALY DETECTION (ISOLATION FOREST)

In [17]:
# Prepare the data for the model

from sklearn.ensemble import IsolationForest

model_data = df[["hemoglobin_g_dl", "wbc_count_10e9_l", "alt_u_l", "systolic_bp_mmhg", "age"]].copy()
model_data = model_data.fillna(model_data.mean())

In [18]:
# Fit the model and get anomaly scores

model = IsolationForest(contamination = 0.05, random_state = 42)
model.fit(model_data)

df["anomaly_score"] = model.decision_function(model_data)
df["is_anomaly"] = model.predict(model_data)

In [19]:


flagged = df[df["is_anomaly"] == -1].sort_values("anomaly_score")

flagged[["subject_id", "site_id", "visit_label", "hemoglobin_g_dl",
         "wbc_count_10e9_l", "alt_u_l", "systolic_bp_mmhg", "age", "anomaly_score"]]

,subject_id,site_id,visit_label,hemoglobin_g_dl,wbc_count_10e9_l,alt_u_l,systolic_bp_mmhg,age,anomaly_score
730,SUBJ-1214,SITE_05,Screening,12.4,10.6,44.0,472.7,38,-0.132339
29,SUBJ-1157,SITE_04,Week4,64.5,10.6,10.1,131.2,21,-0.128952
225,SUBJ-1056,SITE_04,Screening,16.4,10.7,202.9,130.5,59,-0.128102
766,SUBJ-1265,SITE_02,Screening,14.9,4.6,14.6,559.5,37,-0.121345
641,SUBJ-1254,SITE_01,Week4,13.9,NaN,10.9,528.8,72,-0.120531
...,...,...,...,...,...,...,...,...,...
192,SUBJ-1146,SITE_04,Week4,16.5,9.6,55.3,127.7,75,-0.002943
1163,SUBJ-1274,SITE_05,Week12,12.5,5.0,50.0,116.4,19,-0.002071
1118,SUBJ-1027,SITE_03,Week4,16.1,11.0,55.8,117.8,23,-0.001498
1002,SUBJ-1234,SITE_03,Week4,12.6,4.5,42.5,136.4,73,-0.000958


In [20]:
# Compare SQL-flagged vs model-flagged visit-level (row-level)

sql_result = pd.read_sql(query2, conn)
sql_flagged_rows = set(sql_result.apply(lambda r: (r["subject_id"], r["visit_label"]), axis=1))
model_flagged_rows = set(flagged.apply(lambda r: (r["subject_id"], r["visit_label"]), axis=1))

both_rows = sql_flagged_rows & model_flagged_rows
sql_only_rows = sql_flagged_rows - model_flagged_rows
model_only_rows = model_flagged_rows - sql_flagged_rows

print(f"VISIT LEVEL - both: {len(both_rows)}, SQL only: {len(sql_only_rows)}, model only: {len(model_only_rows)}")

# Derive subject-level view FROM the same visit-level (row-level) results
all_flagged_rows = both_rows | sql_only_rows | model_only_rows
subject_visit_counts = pd.Series([pair[0] for pair in all_flagged_rows]).value_counts()

print(f"\nSUBJECT LEVEL - {subject_visit_counts.shape[0]} unique subjects flagged")
print("\nSubjects flagged at more than one visit:")  # Repeat offenders: subjects flagged with out of range                                                    #
subject_visit_counts[subject_visit_counts > 1]       # records at more than one visits



VISIT LEVEL - both: 36, SQL only: 0, model only: 26

SUBJECT LEVEL - 56 unique subjects flagged

Subjects flagged at more than one visit:


SUBJ-1238    3
SUBJ-1025    2
SUBJ-1234    2
SUBJ-1274    2
SUBJ-1275    2
Name: count, dtype: int64

In [21]:
df["is_anomaly"].value_counts()

is_anomaly
 1    1162
-1      62
Name: count, dtype: int64

## PHASE 4: CSV EXPORT FOR POWERBI

In [22]:
df["sql_flagged"] = df.apply(lambda r: (r["subject_id"], r["visit_label"]) in sql_flagged_rows, axis=1)
df["model_flagged"] = df["is_anomaly"] == -1
df["is_duplicate"] = df.duplicated(subset=["subject_id", "visit_label"], keep=False)

dashboard_data = df[["subject_id", "site_id", "visit_label", "visit_date", "age", "sex", 
                     "hemoglobin_g_dl", "wbc_count_10e9_l", "alt_u_l", "systolic_bp_mmhg",
                     "adverse_event_flag", "sql_flagged", "model_flagged", "anomaly_score", "is_duplicate"]]

dashboard_data.to_csv("dashboard_data.csv", index=False)
dashboard_data.head()
dashboard_data.shape


(1224, 15)